# QuantJourney SDK - Factor + Macro Risk Model with Shock Transmission

This notebook demonstrates a QuantJourney SDK workflow that maps macro shocks through factor exposures and volatility context into stress P&L diagnostics.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 4), 'axes.grid': True})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2018-01-01')
END = os.getenv('QJ_EXAMPLE_END', '2026-06-06')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(p):
    v = unwrap(p)
    if isinstance(v, list):
        return v
    if isinstance(v, dict):
        for k in ('rows', 'data', 'items', 'factors'):
            if isinstance(v.get(k), list):
                return v[k]
        return [v]
    return []


## Workflow Helpers

In [ ]:
def get_prices(symbols):
    out = {}
    for s in symbols:
        p = qj.eod.get_historical_prices(symbol=s, start_date=START, end_date=END)
        df = pd.DataFrame(as_rows(p))
        if not df.empty:
            df['date'] = pd.to_datetime(df.get('date'))
            out[s] = pd.to_numeric(df.set_index('date').get('adjusted_close').fillna(df.get('close')), errors='coerce')
    return pd.DataFrame(out).dropna(how='all').sort_index()

def get_ff_factors():
    f = qj.ff.get_factors(region='US')
    rows = as_rows(f)
    if rows:
        df = pd.DataFrame(rows)
        if 'date' in df:
            df = df.set_index(pd.to_datetime(df['date'])).sort_index()
            return df[[c for c in df.columns if c != 'date']].apply(pd.to_numeric, errors='coerce')
    return pd.DataFrame()

def get_macro_shock_proxy():
    for sid in ['DGS10', 'T10Y2Y']:
        m = getattr(qj, 'fred', qj).get_fred_data(series_id=sid, start_date=START, end_date=END)
        rows = as_rows(m)
        if rows:
            d = pd.DataFrame(rows)
            if 'date' in d and 'value' in d:
                s = pd.to_numeric(d['value'], errors='coerce')
                s.index = pd.to_datetime(d['date'])
                return s.diff(21).iloc[-1]
    return 0.25


In [ ]:
assets = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'JPM']
px = get_prices(assets)
ret = px.pct_change().dropna()
ff = get_ff_factors()
common = ret.index.intersection(ff.index)
if len(common) > 100:
    r = ret.loc[common]
    f = ff.loc[common][['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']].fillna(0)
    betas = pd.DataFrame({a: np.linalg.lstsq(f.values, r[a].values, rcond=None)[0] for a in r.columns}, index=f.columns).T
    print('Betas (last window approx):\n', betas.round(3))
shock = get_macro_shock_proxy()
print('\nMacro shock size used:', shock)
if 'Mkt-RF' in locals() or 'betas' in dir():
    mkt_beta = betas.get('Mkt-RF', pd.Series(1.0))
    stress_impact = (mkt_beta * shock * 0.8).round(4)
    print('Approx stress impact per name:\n', stress_impact)
print('\nExtend with full covariance + options surface for hedge sizing. This example already fuses FF + macro + prices.')


## Notes

Multi-source risk example: equity prices + Fama-French factors + macro series for shocks.
All inputs come from the public QuantJourney data layer. Add vol surface / options for realistic hedge suggestions in a follow-up version.
No user portfolios or IBOR used.